In [24]:
import pandas as pd
import numpy as np
import re
import string
import html

import nltk
from nltk import word_tokenize
from langdetect import detect_langs

seed = 123

In [25]:
# Params
MODEL_NAME = "vinai/bertweet-base"
MAX_LEN = 50

In [26]:
# import data and check missing values
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

# rename target col to labels
train = train.rename(columns={"target": "labels"})

# print missing
print("Train missing values:\n", train.isnull().sum())
print("\nTest missing values:\n", test.isnull().sum())

Train missing values:
 id             0
keyword       61
location    2533
text           0
labels         0
dtype: int64

Test missing values:
 id             0
keyword       26
location    1105
text           0
dtype: int64


In [27]:
# clean text, store cleaned as col, "text_cleaned"
def clean_text(text_in:str):
    out = text_in
    out = re.sub("\n", " ", out) # strip multiline

    out = re.sub("B1G", "BIG", out) # strip multiline

    out = re.sub(r"http[^\s]+", " ", out) # strip http* link
    out = re.sub(r'--&gt;', '', out)
    out = out.replace(r'&lt;', '')
    out = out.replace(r'&gt;', '')
    out = out.replace(r'&amp;', '')
    out = re.sub(r"@\w+", "@USER", out) # @___ -> @USER
    out = re.sub(r"[" + re.escape('!"$%&()*+,-./:;<=>?[\\]^_`{|}~') + "]", " ", out) # most punctuation -> space
    out = re.sub(r"\d+", " ", out) # strip numbers
    out = re.sub(" ' ", " ", out) # most punctuation -> space

    # final space processing
    out = re.sub(r"\s+", " ", out) # multispace to 1 space
    out = re.sub(r"^\s+", "", out) # trim leading space
    out = re.sub(r"\s+$", "", out) # trim trailing space

    return out

train["text_cleaned"] = train["text"].apply(clean_text)
test["text_cleaned"] = test["text"].apply(clean_text)

# print before and after
_train_index = None
_test_index = None

# _train_index = 5803
# _test_index = 2125

_tmp_train = train.sample(1) if _train_index is None else train.loc[[_train_index]]
print("[Train] Row Index: ", _tmp_train.index[0])
print("[Train] Before clean: ", _tmp_train["text"].values[0])
print("[Train] After clean: ", _tmp_train["text_cleaned"].values[0])
print("===========================================================")
_tmp_test = test.sample(1) if _test_index is None else test.loc[[_test_index]]
print("[Test] Row Index: ", _tmp_test.index[0])
print("[Test] Before clean: ", _tmp_test["text"].values[0])
print("[Test] After clean: ", _tmp_test["text_cleaned"].values[0])

# clean tmp var
del _tmp_train, _tmp_test, _train_index, _test_index

[Train] Row Index:  4923
[Train] Before clean:  @Mayhem_114 are you reading right to left
[Train] After clean:  @USER are you reading right to left
[Test] Row Index:  314
[Test] Before clean:  Come and join us Tomorrow!
August 7 2015 at Transcend:Blazing the Trail to the Diversified World of Marketing... http://t.co/Fd5od5PfCF
[Test] After clean:  Come and join us Tomorrow August at Transcend Blazing the Trail to the Diversified World of Marketing


In [28]:
# pytorch Dataset Class
import torch
from torch.utils.data import Dataset, DataLoader

class TweetData(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data["text_cleaned"].iloc[idx]
        labels = self.data["labels"].iloc[idx]

        encoded = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

In [ ]:
# load berttweet tokenizer
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, Normalized=True) # Normalized to unify words

In [ ]:
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForSequenceClassification

oof_labels_prob = np.zeros(train.shape[0])

fold = StratifiedKFold(shuffle=True, random_state=123)
for ith_batch, (train_idx, eva_idx) in enumerate(fold.split(X=train, y=train["labels"])):
    _train = train.iloc[train_idx]
    _eval = train.iloc[eva_idx]

    _dataloader_train = DataLoader(TweetData(_train, bert_tokenizer, MAX_LEN), batch_size=32, shuffle=True)
    _dataloader_test = DataLoader(TweetData(_eval, bert_tokenizer, MAX_LEN), batch_size=32, shuffle=True)

    _model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

    break